# Core Python Deep Dive — Senior Developer Interview Prep

A hands-on refresher of intermediate-to-advanced Python fundamentals frequently tested in senior developer interviews.

**Topics Covered:**
1. Advanced Data Structures & Performance
2. Comprehensions — Beyond the Basics
3. Iterators & Generators
4. Decorators
5. Context Managers
6. Closures & Scoping (LEGB)
7. Advanced Error Handling
8. \*args/\*\*kwargs & Unpacking
9. Walrus Operator & Match-Case
10. Practice Problems

In [ ]:
import sys
print(f"Python version: {sys.version}")

---
## 1. Advanced Data Structures & Performance

Understanding *when* to use which data structure is a key differentiator in senior interviews.

### Time Complexity Cheat-Sheet

| Operation | list | dict | set |
|-----------|------|------|-----|
| Lookup by index | O(1) | — | — |
| Lookup by value | O(n) | O(1) | O(1) |
| Insert at end | O(1)* | O(1)* | O(1)* |
| Insert at front | O(n) | — | — |
| Delete by value | O(n) | O(1) | O(1) |

\* amortized

In [6]:
# Real-world example: membership testing — list vs set
import time

data_list = list(range(1_000_000))
data_set = set(data_list)

target = 999_999

start = time.perf_counter()
_ = target in data_list
list_time = time.perf_counter() - start

start = time.perf_counter()
_ = target in data_set
set_time = time.perf_counter() - start

print(f"list lookup: {list_time:.6f}s")
print(f" set lookup: {set_time:.6f}s")
print(f"set is ~{list_time / set_time:.0f}x faster")

list lookup: 0.021537s
 set lookup: 0.000076s
set is ~285x faster


In [7]:
# dict — preserves insertion order (guaranteed since Python 3.7)
# Common interview pattern: using dict as an ordered set
items = [3, 1, 4, 1, 5, 9, 2, 6, 5, 3, 5]
unique_ordered = list(dict.fromkeys(items))
print(f"Original:       {items}")
print(f"Unique ordered: {unique_ordered}")

Original:       [3, 1, 4, 1, 5, 9, 2, 6, 5, 3, 5]
Unique ordered: [3, 1, 4, 5, 9, 2, 6]


In [8]:
# Mutable vs immutable keys — why tuples can be dict keys but lists can't
grid = {}
grid[(0, 0)] = "origin"
grid[(1, 2)] = "point A"
print(grid)

try:
    grid[[1, 2]] = "this fails"  # lists are unhashable
except TypeError as e:
    print(f"Error: {e}")

{(0, 0): 'origin', (1, 2): 'point A'}
Error: unhashable type: 'list'


In [9]:
# Memory comparison: list vs tuple vs set
import sys

data = list(range(1000))
print(f"list:  {sys.getsizeof(data):>6} bytes")
print(f"tuple: {sys.getsizeof(tuple(data)):>6} bytes")
print(f"set:   {sys.getsizeof(set(data)):>6} bytes")
print("\nTuples are immutable → smaller memory footprint, can be used as dict keys.")

list:    8056 bytes
tuple:   8040 bytes
set:    32984 bytes

Tuples are immutable → smaller memory footprint, can be used as dict keys.


In [10]:
# Advanced dict techniques

# 1. dict.setdefault — avoid "if key not in dict" boilerplate
# Real-world: grouping log entries by severity
logs = [
    ("ERROR", "Connection refused"),
    ("INFO", "Server started"),
    ("ERROR", "Timeout"),
    ("WARN", "Disk 80% full"),
    ("INFO", "Request received"),
]

grouped = {}
for level, msg in logs:
    grouped.setdefault(level, []).append(msg)

print(grouped)

{'ERROR': ['Connection refused', 'Timeout'], 'INFO': ['Server started', 'Request received'], 'WARN': ['Disk 80% full']}


In [11]:
# 2. Merging dicts (Python 3.9+)
defaults = {"theme": "dark", "lang": "en", "page_size": 20}
user_prefs = {"theme": "light", "page_size": 50}

merged = defaults | user_prefs  # user_prefs wins on conflicts
print(merged)

# In-place merge
defaults |= user_prefs
print(defaults)

{'theme': 'light', 'lang': 'en', 'page_size': 50}
{'theme': 'light', 'lang': 'en', 'page_size': 50}


---
## 2. Comprehensions — Beyond the Basics

Comprehensions are Pythonic, often faster than loops, and a staple of interview questions.

In [12]:
# List, set, dict comprehensions — quick refresher
squares = [x**2 for x in range(10)]
even_squares = {x**2 for x in range(10) if x % 2 == 0}
word_lengths = {w: len(w) for w in ["python", "java", "go", "rust"]}

print(f"squares:      {squares}")
print(f"even_squares: {even_squares}")
print(f"word_lengths: {word_lengths}")

squares:      [0, 1, 4, 9, 16, 25, 36, 49, 64, 81]
even_squares: {0, 64, 4, 36, 16}
word_lengths: {'python': 6, 'java': 4, 'go': 2, 'rust': 4}


In [ ]:
# Nested comprehensions — flattening a matrix
matrix = [[1, 2, 3], [4, 5, 6], [7, 8, 9]]
flat = [val for row in matrix for val in row]  # outer loop first
print(f"Flattened: {flat}")

# Transpose a matrix
transposed = [[row[i] for row in matrix] for i in range(len(matrix[0]))]
print(f"Transposed: {transposed}")


Flattened: [1, 2, 3, 4, 5, 6, 7, 8, 9]
Transposed: [[1, 4, 7], [2, 5, 8], [3, 6, 9]]


In [15]:
# Real-world: inverting a dict (swap keys and values)
status_codes = {200: "OK", 404: "Not Found", 500: "Server Error"}
inverted = {v: k for k, v in status_codes.items()}
print(inverted)

print(status_codes.items())

{'OK': 200, 'Not Found': 404, 'Server Error': 500}
dict_items([(200, 'OK'), (404, 'Not Found'), (500, 'Server Error')])


In [16]:
# Conditional expressions in comprehensions
# Classify numbers
numbers = range(-5, 6)
classified = [f"{n}:pos" if n > 0 else f"{n}:zero" if n == 0 else f"{n}:neg" for n in numbers]
print(classified)

['-5:neg', '-4:neg', '-3:neg', '-2:neg', '-1:neg', '0:zero', '1:pos', '2:pos', '3:pos', '4:pos', '5:pos']


In [17]:
# Generator expression vs list comprehension — memory
# Generator doesn't store all values in memory
list_comp = [x**2 for x in range(1_000_000)]  # stores 1M items
gen_expr = (x**2 for x in range(1_000_000))   # lazy, stores nothing upfront

print(f"list size: {sys.getsizeof(list_comp):>10} bytes")
print(f" gen size: {sys.getsizeof(gen_expr):>10} bytes")

# Use generator expressions when you only need to iterate once
total = sum(x**2 for x in range(1_000_000))  # no brackets needed inside function
print(f"Sum of squares: {total}")

list size:    8448728 bytes
 gen size:        104 bytes
Sum of squares: 333332833333500000


---
## 3. Iterators & Generators

Understanding the iterator protocol and generators is crucial — they underpin `for` loops, comprehensions, and many standard library functions.

In [18]:
# The Iterator Protocol: __iter__ and __next__

class Countdown:
    """An iterator that counts down from n to 1."""
    def __init__(self, n):
        self.n = n

    def __iter__(self):
        return self

    def __next__(self):
        if self.n <= 0:
            raise StopIteration
        self.n -= 1
        return self.n + 1

for num in Countdown(5):
    print(num, end=" ")
print()

5 4 3 2 1 


In [19]:
# Generator function — same thing, much less code
def countdown(n):
    while n > 0:
        yield n
        n -= 1

print(list(countdown(5)))

# Key insight: yield SUSPENDS the function, preserving local state
gen = countdown(3)
print(next(gen))  # 3 — function runs until first yield
print(next(gen))  # 2 — resumes right after yield
print(next(gen))  # 1

[5, 4, 3, 2, 1]
3
2
1


In [20]:
# Real-world: reading large files line by line without loading into memory
def read_large_file(filepath):
    """Yield one line at a time — works with files larger than RAM."""
    with open(filepath, 'r') as f:
        for line in f:
            yield line.strip()

# Usage: for line in read_large_file('huge.csv'): process(line)

In [21]:
# Generator pipeline — composing generators for data processing
# Real-world scenario: processing a stream of log lines

def generate_logs():
    """Simulate log entries."""
    logs = [
        "2024-01-15 ERROR: Connection timeout",
        "2024-01-15 INFO: User logged in",
        "2024-01-15 ERROR: Database unreachable",
        "2024-01-15 DEBUG: Cache hit",
        "2024-01-15 ERROR: Out of memory",
        "2024-01-15 INFO: Request processed",
    ]
    yield from logs  # yield from delegates to another iterable

def filter_errors(logs):
    """Only pass through ERROR lines."""
    for log in logs:
        if "ERROR" in log:
            yield log

def extract_message(logs):
    """Strip timestamp and level, keep just the message."""
    for log in logs:
        yield log.split(": ", 1)[1]

# Compose the pipeline — nothing executes until we iterate
pipeline = extract_message(filter_errors(generate_logs()))

for msg in pipeline:
    print(f"ALERT: {msg}")


ALERT: Connection timeout
ALERT: Database unreachable
ALERT: Out of memory


In [ ]:
# yield from — delegating to sub-generators
def flatten(nested):
    """Recursively flatten any nested iterable (except strings)."""
    for item in nested:
        if isinstance(item, (list, tuple)) and not isinstance(item, str):
            yield from flatten(item)
        else:
            yield item

nested = [1, [2, 3], [4, [5, 6]], [[7, 8], 9]]
print(list(flatten(nested)))

In [24]:
# send() — two-way communication with generators (coroutine pattern)
def running_average():
    """A generator that maintains a running average.
    Send values in, get the current average back."""
    total = 0
    count = 0
    average = None
    while True:
        value = yield average  # receive a value, yield the average
        if value is not None:
            total += value
            count += 1
            average = total / count

avg = running_average()
next(avg)  # prime the generator (advance to first yield)

print(avg.send(10))  # 10.0
print(avg.send(20))  # 15.0
print(avg.send(30))  # 20.0
print(avg.send(40))  # 25.0

10.0
15.0
20.0
25.0


---
## 4. Decorators

Decorators are syntactic sugar for higher-order functions. They modify or extend function/class behavior without changing source code. A **must-know** for senior interviews.

In [ ]:
# Basic decorator anatomy
import functools

def my_decorator(func):
    @functools.wraps(func)  # preserves __name__, __doc__, etc.
    def wrapper(*args, **kwargs):
        print(f"Calling {func.__name__}")
        result = func(*args, **kwargs)
        print(f"{func.__name__} returned {result}")
        return result
    return wrapper

@my_decorator
def add(a, b):
    """Add two numbers."""
    return a + b

add(3, 4)
print(f"Function name preserved: {add.__name__}")
print(f"Docstring preserved: {add.__doc__}")

In [ ]:
# Real-world decorator: timing function execution
import time
import functools

def timer(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        elapsed = time.perf_counter() - start
        print(f"{func.__name__} took {elapsed:.4f}s")
        return result
    return wrapper

@timer
def slow_function():
    time.sleep(0.1)
    return "done"

slow_function()

In [ ]:
# Real-world decorator: retry with exponential backoff
import functools
import time
import random

def retry(max_attempts=3, backoff_factor=2):
    """Parametrized decorator — returns a decorator."""
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            for attempt in range(1, max_attempts + 1):
                try:
                    return func(*args, **kwargs)
                except Exception as e:
                    if attempt == max_attempts:
                        raise
                    wait = backoff_factor ** attempt
                    print(f"Attempt {attempt} failed: {e}. Retrying in {wait}s...")
                    time.sleep(wait * 0.01)  # shortened for demo
        return wrapper
    return decorator

@retry(max_attempts=3, backoff_factor=2)
def unreliable_api_call():
    if random.random() < 0.7:
        raise ConnectionError("Service unavailable")
    return {"status": "success"}

try:
    result = unreliable_api_call()
    print(f"Result: {result}")
except ConnectionError:
    print("All retries exhausted")

In [1]:
# Real-world decorator: caching / memoization
import functools

def memoize(func):
    """Simple memoization decorator."""
    cache = {}
    @functools.wraps(func)
    def wrapper(*args):
        if args not in cache:
            cache[args] = func(*args)
        return cache[args]
    wrapper.cache = cache  # expose cache for inspection
    return wrapper

@memoize
def fibonacci(n):
    if n < 2:
        return n
    return fibonacci(n - 1) + fibonacci(n - 2)

print(f"fib(50) = {fibonacci(50)}")
print(f"Cache size: {len(fibonacci.cache)}")

# In practice, use functools.lru_cache:
@functools.lru_cache(maxsize=128)
def fibonacci_v2(n):
    if n < 2:
        return n
    return fibonacci_v2(n - 1) + fibonacci_v2(n - 2)

print(f"fib_v2(100) = {fibonacci_v2(100)}")
print(f"Cache info: {fibonacci_v2.cache_info()}")

fib(50) = 12586269025
Cache size: 51
fib_v2(100) = 354224848179261915075
Cache info: CacheInfo(hits=98, misses=101, maxsize=128, currsize=101)


In [ ]:
# Stacking decorators — execution order
def bold(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        return f"<b>{func(*args, **kwargs)}</b>"
    return wrapper

def italic(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        return f"<i>{func(*args, **kwargs)}</i>"
    return wrapper

@bold
@italic
def greet(name):
    return f"Hello, {name}"

# Equivalent to: bold(italic(greet))
# italic runs first (innermost), bold wraps that
print(greet("World"))  # <b><i>Hello, World</i></b>

In [ ]:
# Class-based decorator with state
import functools

class CallCounter:
    """Tracks how many times a function has been called."""
    def __init__(self, func):
        functools.update_wrapper(self, func)
        self.func = func
        self.count = 0

    def __call__(self, *args, **kwargs):
        self.count += 1
        return self.func(*args, **kwargs)

@CallCounter
def process_data(data):
    return [x * 2 for x in data]

process_data([1, 2, 3])
process_data([4, 5, 6])
process_data([7, 8, 9])
print(f"process_data called {process_data.count} times")

---
## 5. Context Managers

Context managers ensure resources are properly acquired and released. They power the `with` statement and are essential for writing robust, leak-free code.

In [ ]:
# The Context Manager Protocol: __enter__ and __exit__

class Timer:
    """Time a block of code using a with statement."""
    def __enter__(self):
        self.start = time.perf_counter()
        return self  # returned value is bound to the 'as' variable

    def __exit__(self, exc_type, exc_val, exc_tb):
        self.elapsed = time.perf_counter() - self.start
        print(f"Elapsed: {self.elapsed:.4f}s")
        return False  # don't suppress exceptions

with Timer() as t:
    total = sum(range(1_000_000))

print(f"Stored elapsed: {t.elapsed:.4f}s")

In [ ]:
# contextlib.contextmanager — write context managers as generators
from contextlib import contextmanager

@contextmanager
def temporary_directory():
    """Create a temp dir, yield it, clean up afterward."""
    import tempfile, shutil
    path = tempfile.mkdtemp()
    print(f"Created temp dir: {path}")
    try:
        yield path  # everything before yield = __enter__
    finally:
        shutil.rmtree(path)  # everything after yield = __exit__
        print(f"Cleaned up temp dir: {path}")

with temporary_directory() as tmpdir:
    print(f"Working in: {tmpdir}")
    # temp dir is auto-cleaned after this block

In [ ]:
# Real-world: database transaction context manager
@contextmanager
def db_transaction(connection):
    """Commit on success, rollback on failure."""
    try:
        yield connection
        connection.commit()
        print("Transaction committed")
    except Exception:
        connection.rollback()
        print("Transaction rolled back")
        raise

# Simulated connection
class FakeConnection:
    def commit(self): pass
    def rollback(self): pass

with db_transaction(FakeConnection()) as conn:
    print("Executing queries...")

In [ ]:
# Exception suppression: __exit__ returning True
class SuppressErrors:
    def __init__(self, *exceptions):
        self.exceptions = exceptions

    def __enter__(self):
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        if exc_type and issubclass(exc_type, self.exceptions):
            print(f"Suppressed: {exc_type.__name__}: {exc_val}")
            return True  # suppress the exception
        return False

with SuppressErrors(FileNotFoundError, PermissionError):
    open("nonexistent_file.txt")  # FileNotFoundError suppressed

print("Execution continues normally!")

# stdlib equivalent: contextlib.suppress
from contextlib import suppress
with suppress(FileNotFoundError):
    open("nonexistent_file.txt")

---
## 6. Closures & Scoping (LEGB)

Python resolves variable names using the **LEGB** rule:  
**L**ocal → **E**nclosing → **G**lobal → **B**uilt-in

Closures happen when an inner function captures variables from its enclosing scope.

In [ ]:
# LEGB in action
x = "global"

def outer():
    x = "enclosing"
    def inner():
        x = "local"
        print(f"inner sees: {x}")   # local
    inner()
    print(f"outer sees: {x}")       # enclosing

outer()
print(f"module sees: {x}")          # global

In [ ]:
# Closures — the inner function "closes over" enclosing variables

def make_multiplier(factor):
    """Factory function that returns a closure."""
    def multiplier(x):
        return x * factor  # 'factor' is captured from enclosing scope
    return multiplier

double = make_multiplier(2)
triple = make_multiplier(3)

print(f"double(5) = {double(5)}")  # 10
print(f"triple(5) = {triple(5)}")  # 15

# Inspect the closure
print(f"Closed-over vars: {double.__closure__[0].cell_contents}")

In [ ]:
# CLASSIC GOTCHA: late binding in closures
# This is a very common interview question!

functions = []
for i in range(5):
    functions.append(lambda: i)  # all lambdas reference the SAME 'i'

print("Late binding (wrong):")
print([f() for f in functions])  # [4, 4, 4, 4, 4] — all see i=4!

# Fix 1: default argument (captures value at definition time)
functions = []
for i in range(5):
    functions.append(lambda i=i: i)  # 'i=i' captures current value

print("\nDefault arg fix:")
print([f() for f in functions])  # [0, 1, 2, 3, 4]

# Fix 2: use a factory function
def make_func(n):
    return lambda: n

functions = [make_func(i) for i in range(5)]
print("\nFactory fix:")
print([f() for f in functions])  # [0, 1, 2, 3, 4]

In [ ]:
# nonlocal — modify enclosing scope variable
def counter():
    count = 0
    def increment():
        nonlocal count  # without this, assignment creates a new local
        count += 1
        return count
    return increment

c = counter()
print(c(), c(), c())  # 1 2 3

---
## 7. Advanced Error Handling

Senior developers should know more than `try/except`. Understanding exception hierarchies, chaining, and custom exceptions shows depth.

In [ ]:
# Custom exception hierarchy — real-world API client

class APIError(Exception):
    """Base exception for API operations."""
    def __init__(self, message, status_code=None):
        super().__init__(message)
        self.status_code = status_code

class AuthenticationError(APIError):
    """Invalid or expired credentials."""
    pass

class RateLimitError(APIError):
    """Too many requests."""
    def __init__(self, message, retry_after=60):
        super().__init__(message, status_code=429)
        self.retry_after = retry_after

class NotFoundError(APIError):
    pass

# Usage
try:
    raise RateLimitError("Slow down!", retry_after=30)
except RateLimitError as e:
    print(f"Rate limited: {e}. Retry in {e.retry_after}s")
except APIError as e:
    print(f"API error: {e}")

In [2]:
# Exception chaining — 'from' preserves the original cause

def load_config(path):
    try:
        with open(path) as f:
            return f.read()
    except FileNotFoundError as e:
        raise RuntimeError(f"Config file missing: {path}") from e

try:
    load_config("missing.yaml")
except RuntimeError as e:
    print(f"Error: {e}")
    print(f"Caused by: {e.__cause__}")

Error: Config file missing: missing.yaml
Caused by: [Errno 2] No such file or directory: 'missing.yaml'


In [3]:
# else and finally — often misunderstood

def divide(a, b):
    try:
        result = a / b
    except ZeroDivisionError:
        print("Cannot divide by zero!")
        return None
    else:
        # runs ONLY if no exception was raised
        print(f"{a}/{b} = {result}")
        return result
    finally:
        # ALWAYS runs — even if there's a return
        print("Division attempt complete.")

divide(10, 3)
print()
divide(10, 0)

10/3 = 3.3333333333333335
Division attempt complete.

Cannot divide by zero!
Division attempt complete.


In [ ]:
# ExceptionGroup (Python 3.11+) — handling multiple simultaneous errors
import sys

if sys.version_info >= (3, 11):
    def validate_user(data):
        errors = []
        if not data.get("name"):
            errors.append(ValueError("Name is required"))
        if not data.get("email") or "@" not in data.get("email", ""):
            errors.append(ValueError("Valid email is required"))
        if data.get("age", 0) < 18:
            errors.append(ValueError("Must be 18 or older"))
        if errors:
            raise ExceptionGroup("Validation failed", errors)
        return True

    try:
        validate_user({"name": "", "email": "bad", "age": 15})
    except* ValueError as eg:
        for err in eg.exceptions:
            print(f"  - {err}")
else:
    print(f"Python {sys.version_info.major}.{sys.version_info.minor} — ExceptionGroup requires 3.11+")

---
## 8. \*args / \*\*kwargs & Unpacking

Flexible function signatures and unpacking operators are used everywhere in Python — frameworks, decorators, API wrappers.

In [4]:
# *args and **kwargs
def flexible_func(*args, **kwargs):
    print(f"Positional args: {args}")
    print(f"Keyword args: {kwargs}")

flexible_func(1, 2, 3, name="Alice", age=30)

Positional args: (1, 2, 3)
Keyword args: {'name': 'Alice', 'age': 30}


In [ ]:
# Keyword-only arguments (after *)
def create_user(name, *, email, role="viewer"):
    """email and role MUST be passed as keyword arguments."""
    return {"name": name, "email": email, "role": role}

print(create_user("Alice", email="alice@example.com", role="admin"))

try:
    create_user("Bob", "bob@example.com")  # fails — email is keyword-only
except TypeError as e:
    print(f"Error: {e}")

In [ ]:
# Positional-only arguments (Python 3.8+, before /)
def pow(base, exp, /):
    """base and exp can only be passed positionally."""
    return base ** exp

print(pow(2, 10))  # 1024

try:
    pow(base=2, exp=10)  # fails
except TypeError as e:
    print(f"Error: {e}")

In [ ]:
# Advanced unpacking

# Extended unpacking with *
first, *middle, last = [1, 2, 3, 4, 5]
print(f"first={first}, middle={middle}, last={last}")

# Swap without temp variable
a, b = 1, 2
a, b = b, a
print(f"After swap: a={a}, b={b}")

# Nested unpacking
(a, b), (c, d) = [1, 2], [3, 4]
print(f"a={a}, b={b}, c={c}, d={d}")

# Dict unpacking for function calls
config = {"host": "localhost", "port": 5432, "database": "mydb"}
def connect(host, port, database):
    return f"Connecting to {database}@{host}:{port}"

print(connect(**config))

---
## 9. Walrus Operator & Match-Case

Modern Python features that demonstrate up-to-date knowledge.

In [ ]:
# Walrus operator (:=) — assignment expression (Python 3.8+)
# Assign and use a value in the same expression

# Without walrus
data = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
filtered = []
for x in data:
    y = x ** 2
    if y > 20:
        filtered.append(y)

# With walrus — compute once, use in both condition and result
filtered_walrus = [y for x in data if (y := x ** 2) > 20]

print(f"Filtered: {filtered}")
print(f"Walrus:   {filtered_walrus}")

In [ ]:
# Walrus in while loops — common pattern for input/read loops
import io

fake_file = io.StringIO("line 1\nline 2\nline 3\n")

while (line := fake_file.readline().strip()):
    print(f"Read: {line}")

In [ ]:
# match-case — structural pattern matching (Python 3.10+)
import sys

if sys.version_info >= (3, 10):
    def handle_command(command):
        match command.split():
            case ["quit" | "exit"]:
                return "Goodbye!"
            case ["hello", name]:
                return f"Hello, {name}!"
            case ["add", *numbers] if all(n.isdigit() for n in numbers):
                return f"Sum: {sum(int(n) for n in numbers)}"
            case ["move", ("up" | "down" | "left" | "right") as direction]:
                return f"Moving {direction}"
            case _:
                return f"Unknown command: {command}"

    print(handle_command("hello Alice"))
    print(handle_command("add 1 2 3 4 5"))
    print(handle_command("move up"))
    print(handle_command("quit"))
    print(handle_command("something else"))
else:
    print(f"Python {sys.version_info.major}.{sys.version_info.minor} — match-case requires 3.10+")

In [ ]:
# match-case with class patterns
if sys.version_info >= (3, 10):
    from dataclasses import dataclass

    @dataclass
    class Point:
        x: float
        y: float

    def classify_point(point):
        match point:
            case Point(x=0, y=0):
                return "origin"
            case Point(x=0, y=y):
                return f"on y-axis at y={y}"
            case Point(x=x, y=0):
                return f"on x-axis at x={x}"
            case Point(x=x, y=y) if x > 0 and y > 0:
                return f"quadrant I ({x}, {y})"
            case _:
                return f"somewhere at ({point.x}, {point.y})"

    for p in [Point(0, 0), Point(0, 5), Point(3, 0), Point(1, 2), Point(-1, -2)]:
        print(f"{p} → {classify_point(p)}")
else:
    print("Requires Python 3.10+")

---
## 10. Practice Problems

Try these yourself before revealing the solutions. These are commonly asked in senior Python interviews.

### Problem 1: Write a `@validate_types` decorator

Write a decorator that validates function argument types using annotations.

```python
@validate_types
def greet(name: str, times: int) -> str:
    return name * times

greet("hi", 3)    # works
greet("hi", "3")  # raises TypeError
```

In [ ]:
# YOUR SOLUTION HERE


In [ ]:
# SOLUTION
import functools
import inspect

def validate_types(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        sig = inspect.signature(func)
        bound = sig.bind(*args, **kwargs)
        bound.apply_defaults()

        hints = func.__annotations__
        for param_name, value in bound.arguments.items():
            if param_name in hints and not isinstance(value, hints[param_name]):
                raise TypeError(
                    f"Argument '{param_name}' expected {hints[param_name].__name__}, "
                    f"got {type(value).__name__}"
                )
        return func(*args, **kwargs)
    return wrapper

@validate_types
def greet(name: str, times: int) -> str:
    return name * times

print(greet("Hello! ", 3))

try:
    greet("hi", "3")
except TypeError as e:
    print(f"Caught: {e}")

### Problem 2: Implement a generator-based `chunked` function

Given an iterable and a chunk size, yield successive chunks.

```python
list(chunked([1,2,3,4,5,6,7], 3))  # [[1,2,3], [4,5,6], [7]]
list(chunked("abcdefg", 2))         # ['ab', 'cd', 'ef', 'g']
```

In [ ]:
# YOUR SOLUTION HERE


In [ ]:
# SOLUTION
from itertools import islice

def chunked(iterable, size):
    """Yield successive chunks from iterable."""
    it = iter(iterable)
    while True:
        chunk = list(islice(it, size))
        if not chunk:
            break
        yield chunk

print(list(chunked([1, 2, 3, 4, 5, 6, 7], 3)))
print(list(chunked(range(10), 4)))
print(list(chunked("abcdefg", 2)))

### Problem 3: Build a `@rate_limit` context manager

Create a context manager that limits how many times a block can execute per second.

```python
limiter = RateLimiter(max_calls=2, period=1.0)  # 2 calls per second
for i in range(5):
    with limiter:
        print(f"Call {i}")
```

In [ ]:
# YOUR SOLUTION HERE


In [ ]:
# SOLUTION
import time
from collections import deque

class RateLimiter:
    def __init__(self, max_calls, period=1.0):
        self.max_calls = max_calls
        self.period = period
        self.calls = deque()

    def __enter__(self):
        now = time.monotonic()
        # Remove timestamps outside the window
        while self.calls and self.calls[0] <= now - self.period:
            self.calls.popleft()
        # If at capacity, sleep until oldest call exits the window
        if len(self.calls) >= self.max_calls:
            sleep_time = self.period - (now - self.calls[0])
            if sleep_time > 0:
                time.sleep(sleep_time)
            self.calls.popleft()
        self.calls.append(time.monotonic())
        return self

    def __exit__(self, *args):
        return False

# Demo (limited to 3 calls/sec)
limiter = RateLimiter(max_calls=3, period=1.0)
start = time.perf_counter()
for i in range(6):
    with limiter:
        elapsed = time.perf_counter() - start
        print(f"Call {i} at {elapsed:.2f}s")

### Problem 4: Implement a lazy property decorator

Create a descriptor that computes a property value only once and caches it.

```python
class DataProcessor:
    @lazy_property
    def data(self):
        print("Computing...")
        return expensive_operation()
```

In [ ]:
# YOUR SOLUTION HERE


In [ ]:
# SOLUTION
class lazy_property:
    """A descriptor that replaces itself with the computed value on first access."""
    def __init__(self, func):
        self.func = func
        self.attr_name = func.__name__

    def __get__(self, obj, objtype=None):
        if obj is None:
            return self
        value = self.func(obj)
        setattr(obj, self.attr_name, value)  # replace descriptor with value
        return value

class DataProcessor:
    def __init__(self, raw):
        self.raw = raw

    @lazy_property
    def processed(self):
        print("Expensive computation happening...")
        return [x ** 2 for x in self.raw]

dp = DataProcessor([1, 2, 3, 4, 5])
print("First access:")
print(dp.processed)  # triggers computation
print("\nSecond access:")
print(dp.processed)  # uses cached value, no "Expensive computation" message

# Note: Python 3.8+ has functools.cached_property for this

### Problem 5: What does this code print? (Closure + Mutable Default)

Predict the output without running it, then verify.

```python
def make_adders():
    adders = []
    for i in range(3):
        def adder(x, memo=[]):
            memo.append(x + i)
            return memo
        adders.append(adder)
    return adders

fns = make_adders()
print(fns[0](10))
print(fns[1](20))
print(fns[2](30))
```

In [ ]:
# Think about it first, then run to verify

def make_adders():
    adders = []
    for i in range(3):
        def adder(x, memo=[]):
            memo.append(x + i)
            return memo
        adders.append(adder)
    return adders

fns = make_adders()
print(fns[0](10))  # ?
print(fns[1](20))  # ?
print(fns[2](30))  # ?

# Explanation:
# 1. Late binding: all adders see i=2 (final loop value)
# 2. Mutable default: ALL adders share the SAME list []
# So each call appends to the same list with i=2:
# fns[0](10) → [12]       (10+2)
# fns[1](20) → [12, 22]   (20+2 appended to same list)
# fns[2](30) → [12, 22, 32] (30+2 appended to same list)

### Problem 6: Write a `@singleton` decorator for classes

```python
@singleton
class Database:
    def __init__(self, url):
        self.url = url

db1 = Database("postgres://localhost")
db2 = Database("postgres://localhost")
assert db1 is db2  # same instance
```

In [ ]:
# YOUR SOLUTION HERE


In [ ]:
# SOLUTION
def singleton(cls):
    instances = {}
    @functools.wraps(cls)
    def get_instance(*args, **kwargs):
        if cls not in instances:
            instances[cls] = cls(*args, **kwargs)
        return instances[cls]
    return get_instance

@singleton
class Database:
    def __init__(self, url):
        self.url = url
        print(f"Database created with {url}")

db1 = Database("postgres://localhost")  # prints "Database created..."
db2 = Database("postgres://localhost")  # no print — returns cached instance

print(f"Same instance? {db1 is db2}")
print(f"URL: {db1.url}")